# Frigate / native SatSim / SPOT — conditioned image-formation comparison

Compare SPOT with functions imported from an installed SatSim package using common detector-plane source maps. Run from the `python/` directory in an environment with SatSim and its TensorFlow dependencies. Set `SATSIM_SOURCE_DIR` only when using a local SatSim source checkout. The optional observation files must be present in `Inputs/Validation/`; see that directory's README.

The inputs comprise a Frigate FITS observation, solved WCS, MAT star catalogue, 4 × 4 arithmetic block means, a fixed ROI, six measured trail endpoint pairs, empirical background and amplitude conditioning.

**SatSim execution:** `satsim.image.psf.gen_gaussian`, `satsim.math.fft.fftconv2`, and `satsim.image.noise.add_photon_noise`. The Gaussian is sampled at the working pixel scale, truncated at ±ceil(4σ), and normalized to unit sum. Convolution uses zero exterior support. The empirical diffuse background is added after the source PSF. A common effective gain converts DN-equivalent inputs to electrons before photon-noise sampling; output is fractional DN.

**Scope:** this comparison exercises native optical and photon-noise APIs, rather than the SatSim `gen_multi` scene generator. Source allocation and target shapes are shared conditioning inputs. SatSim's integer ADC is outside this comparison. Dark current, read noise, bias, vignetting and saturation are not additional fitted effects in the SatSim path. The SPOT deterministic response is checked against the common optical model before metrics are calculated.

The observation supplies both conditioning inputs and evaluation data. Results assess detector-plane consistency and agreement with that observation; they do not independently validate orbit propagation, camera geometry, absolute radiometry or target surface photometry.

Outputs include noise-off/noise-on arrays, comparison tables, a six-panel figure, diagnostics, input hashes, package/source hashes and run settings.


## 1. Paths, environment and required packages


In [ ]:
# EDIT THESE TWO VALUES ONLY IF NEEDED.
from pathlib import Path
SPOT_ROOT = None  # Detect the python project folder from the working directory.
SATSIM_SOURCE_DIR = None  # Optional path to a SatSim source checkout.
# SATSIM_SOURCE_DIR is the OUTER folder containing the inner satsim package.
# Leave None to use the SatSim installation already available to this kernel.
USE_CPU = True

import os, sys, copy, hashlib, inspect, json, platform, subprocess
from datetime import datetime, timezone
from importlib import metadata as package_metadata

if USE_CPU:
    os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # effective before TensorFlow import

if SPOT_ROOT is None:
    candidates = [Path.cwd(), *Path.cwd().parents]
    ROOT = next((p for p in candidates if (p / 'ssa_emulator').is_dir()), None)
    if ROOT is None:
        raise FileNotFoundError(
            'Open this notebook in the python folder, or set SPOT_ROOT '
            'to the folder containing ssa_emulator and Inputs, then rerun.'
        )
else:
    ROOT = Path(SPOT_ROOT).expanduser().resolve()
    if not (ROOT / 'ssa_emulator').is_dir():
        raise FileNotFoundError(f'ssa_emulator was not found under {ROOT}')

sys.path.insert(0, str(ROOT))
if SATSIM_SOURCE_DIR is not None:
    SATSIM_SOURCE_DIR = Path(SATSIM_SOURCE_DIR).expanduser().resolve()
    if not (SATSIM_SOURCE_DIR / 'satsim' / '__init__.py').is_file():
        raise FileNotFoundError('SATSIM_SOURCE_DIR must contain satsim/__init__.py')
    sys.path.insert(0, str(SATSIM_SOURCE_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import binary_dilation
from IPython.display import display

try:
    import tensorflow as tf
    import satsim
    from satsim.image.psf import gen_gaussian as satsim_gen_gaussian
    from satsim.math.fft import fftconv2 as satsim_fftconv2
    from satsim.image.noise import add_photon_noise as satsim_add_photon_noise
except ImportError as exc:
    raise ImportError(
        'Install SatSim and its TensorFlow dependencies in this kernel environment. '
        'For a local source checkout, set SATSIM_SOURCE_DIR above. '
        'This notebook requires native SatSim APIs.'
    ) from exc

if USE_CPU:
    try:
        tf.config.set_visible_devices([], 'GPU')
    except RuntimeError as exc:
        raise RuntimeError('Restart the kernel and Run All so CPU selection takes effect.') from exc
if not tf.executing_eagerly():
    raise RuntimeError('This notebook requires TensorFlow eager execution.')

SATSIM_PACKAGE_DIR = Path(satsim.__file__).resolve().parent
if SATSIM_SOURCE_DIR is not None and SATSIM_PACKAGE_DIR != SATSIM_SOURCE_DIR / 'satsim':
    raise RuntimeError('Another SatSim copy is cached. Restart the kernel, then Run All.')

import ssa_emulator
from ssa_emulator.config import default_config
from ssa_emulator.frigate_conditioning import (
    apply_spot_to_conditioned_maps, background_residual,
    configure_spot_for_detector_maps, downsample_image,
    extract_frigate_metadata, extract_wcs_metadata, load_fits_image,
    load_solved_wcs, manual_trails_from_roi, project_catalogue_to_detector,
    render_star_flux, render_trail_flux, scale_roi, load_validation_catalogue,
)
from ssa_emulator.validation import (
    direct_pair_metrics, target_integrated_flux, frigate_reference_metrics,
)
from ssa_emulator.export import json_safe

if Path(ssa_emulator.__file__).resolve().parent != ROOT / 'ssa_emulator':
    raise RuntimeError('Another SPOT copy is cached. Restart the kernel, then Run All.')

DATA_DIR = ROOT / 'Inputs' / 'Validation'
RAW_FITS = DATA_DIR / 'Capture_00938 03_05_42Z.fits'
WCS_FITS = DATA_DIR / 'wcs.fits'
SOLVED_FITS = DATA_DIR / 'new-image.fits'
CATALOGUE = DATA_DIR / 'StarDB_500k.mat'
for required in (RAW_FITS, CATALOGUE):
    if not required.is_file():
        raise FileNotFoundError(f'Update the input path or restore this file: {required}')
if not WCS_FITS.is_file() and not SOLVED_FITS.is_file():
    raise FileNotFoundError('Provide either wcs.fits or new-image.fits in DATA_DIR.')

OUT_DIR = ROOT / 'Outputs' / ('validation_actual_satsim_' +
                             datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S_%f'))
OUT_DIR.mkdir(parents=True, exist_ok=False)
DOWNSAMPLE = 4
ROI_FULL = (6300, 5440, 7300, 6260)
ROI_WORK = scale_roi(ROI_FULL, DOWNSAMPLE)
SPOT_SEED, SATSIM_SEED = 12, 21
FITTED_PSF_SIGMA_PX = 0.75
SATSIM_TILE_SIZE = 256  # finite-kernel halo makes tile interiors equivalent to a full convolution

print('Python:', sys.executable)
print('SPOT:', Path(ssa_emulator.__file__).resolve())
print('SatSim:', satsim.__file__)
print('SatSim imported version:', getattr(satsim, '__version__', 'not exposed'))
print('TensorFlow:', tf.__version__)
print('Output:', OUT_DIR)


## 2. Native SatSim functions and small numerical checks

These checks run on synthetic arrays only. They verify pixel registration, flux conservation away from the detector edge, and the mean/variance of the native photon-noise routine. They are not Frigate validation results.


In [ ]:
NATIVE_FUNCTIONS = {
    'gaussian_psf': satsim_gen_gaussian,
    'linear_convolution': satsim_fftconv2,
    'photon_noise': satsim_add_photon_noise,
}

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def native_function_record(func):
    path = Path(inspect.getsourcefile(func)).resolve()
    if not path.is_relative_to(SATSIM_PACKAGE_DIR):
        raise RuntimeError(f'{func.__name__} is not loaded from the imported SatSim package.')
    return {
        'module': func.__module__, 'function': func.__name__,
        'signature': str(inspect.signature(func)), 'source_file': str(path),
        'source_sha256': sha256_file(path),
    }

native_api_provenance = {name: native_function_record(func)
                         for name, func in NATIVE_FUNCTIONS.items()}

def make_satsim_kernel(sigma_px):
    if not np.isfinite(sigma_px) or sigma_px <= 0:
        raise ValueError('PSF sigma must be finite and positive.')
    radius = int(np.ceil(4.0 * sigma_px))
    side = 2 * radius + 1
    raw = np.asarray(satsim_gen_gaussian(side, side, float(sigma_px)).numpy(),
                     dtype=np.float64)
    raw_sum = float(raw.sum())
    if not np.isfinite(raw_sum) or raw_sum <= 0:
        raise RuntimeError('SatSim returned an invalid Gaussian PSF.')
    return raw / raw_sum, {'sigma_px': float(sigma_px), 'radius_px': radius,
                          'raw_sampled_sum': raw_sum, 'normalization': 'unit discrete sum'}

SATSIM_KERNEL, satsim_psf_settings = make_satsim_kernel(FITTED_PSF_SIGMA_PX)

def validate_map(value, name, shape=None):
    a = np.asarray(value, dtype=np.float64)
    if a.ndim != 2 or (shape is not None and a.shape != tuple(shape)):
        raise ValueError(f'{name}: unexpected shape {a.shape}; expected {shape}')
    if not np.isfinite(a).all() or np.any(a < 0):
        raise ValueError(f'{name} must contain finite, nonnegative values.')
    return a

def satsim_convolve_map(point_map, kernel=SATSIM_KERNEL, tile_size=SATSIM_TILE_SIZE):
    """Native SatSim linear FFT convolution, using halo tiles to bound memory.

    Inputs are already allocated to detector pixels. This adds no resampling,
    motion integration, exposure multiplication or fitted image registration.
    Odd PSF support gives an integer-pixel centre on both odd/even detector shapes.
    """
    a = validate_map(point_map, 'point-source map')
    if kernel.ndim != 2 or kernel.shape[0] != kernel.shape[1] or kernel.shape[0] % 2 != 1:
        raise ValueError('An odd, square PSF kernel is required.')
    radius = kernel.shape[0] // 2
    height, width = a.shape
    result = np.zeros_like(a)
    padded = np.pad(a, ((radius, radius), (radius, radius)), mode='constant')
    psfs = {}
    for row in range(0, height, tile_size):
        for col in range(0, width, tile_size):
            h = min(tile_size, height - row)
            w = min(tile_size, width - col)
            patch = padded[row:row + h + 2 * radius, col:col + w + 2 * radius]
            if not np.any(patch):
                continue
            # fftconv2 requires equal-sized source/PSF arrays. Odd support
            # avoids the half-pixel centre of an even-sized Gaussian array.
            fft_shape = (patch.shape[0] | 1, patch.shape[1] | 1)
            image_tf = tf.convert_to_tensor(np.pad(
                patch, ((0, fft_shape[0] - patch.shape[0]),
                        (0, fft_shape[1] - patch.shape[1]))), dtype=tf.float64)
            if fft_shape not in psfs:
                psf = np.zeros(fft_shape, dtype=np.float64)
                r, c = fft_shape[0] // 2, fft_shape[1] // 2
                psf[r-radius:r+radius+1, c-radius:c+radius+1] = kernel
                psfs[fft_shape] = tf.convert_to_tensor(psf, dtype=tf.float64)
            convolved = satsim_fftconv2(
                image_tf, psfs[fft_shape], dtype=tf.complex128).numpy()
            result[row:row+h, col:col+w] = convolved[radius:radius+h, radius:radius+w]
    tolerance = 1e-10 * max(1.0, float(a.max()))
    if float(result.min()) < -tolerance:
        raise RuntimeError('Unexpected negative convolution values; check the SatSim API.')
    return np.maximum(result, 0.0)  # remove numerical FFT round-off only

smoke_checks = []
for shape in ((31, 34), (32, 33)):
    for position in ((14, 16), (0, 0)):
        test = np.zeros(shape, dtype=float)
        test[position] = 1.0
        actual = satsim_convolve_map(test, tile_size=11)
        expected = np.zeros(shape, dtype=float)
        rad = SATSIM_KERNEL.shape[0] // 2
        for ir in range(-rad, rad + 1):
            for ic in range(-rad, rad + 1):
                r, c = position[0] + ir, position[1] + ic
                if 0 <= r < shape[0] and 0 <= c < shape[1]:
                    expected[r, c] = SATSIM_KERNEL[ir + rad, ic + rad]
        np.testing.assert_allclose(actual, expected, atol=2e-8, rtol=2e-7)
        smoke_checks.append({'shape': list(shape), 'impulse_position': list(position),
                             'max_error': float(np.max(np.abs(actual - expected))),
                             'retained_flux': float(actual.sum())})

tf.random.set_seed(913)
test_e = np.asarray(satsim_add_photon_noise(
    tf.fill([256, 256], tf.constant(2000.0, tf.float32)), samples=None).numpy())
test_mean, test_variance = float(test_e.mean()), float(test_e.var())
if abs(test_mean - 2000.0) > 2.0 or abs(test_variance - 2000.0) > 200.0:
    raise RuntimeError('Native photon-noise check failed: inspect the SatSim/TensorFlow versions.')
native_smoke_results = {'convolution': smoke_checks,
                        'poisson_test_mean_e': test_mean,
                        'poisson_test_variance_e2': test_variance}
print('Native SatSim checks passed: PSF registration, retained flux, and photon noise.')
display(pd.DataFrame(native_api_provenance).T[['module', 'function']])


## 3. Load measured FITS/WCS and the trail signatures

Arithmetic 4 × 4 block means are retained; incomplete edge blocks are discarded. The six endpoint pairs are zero-based coordinates within the 205 × 250 ROI.


In [ ]:
# Load FITS & WCS
frigate_full, raw_header = load_fits_image(RAW_FITS)
frigate = downsample_image(frigate_full, DOWNSAMPLE)
meta = extract_frigate_metadata(RAW_FITS, frigate_full, raw_header)
wcs_path = WCS_FITS if WCS_FITS.exists() else SOLVED_FITS
wcs = load_solved_wcs(wcs_path)
wcs_meta = extract_wcs_metadata(wcs, frigate_full.shape)

x0, y0, x1, y1 = ROI_WORK
frigate_roi = frigate[y0:y1, x0:x1]

# Trail signatures (Local ROI endpoints)
manual_trails_roi = pd.DataFrame(
    [
        [1, 55, 156, 59, 128, 4.0],
        [2, 65, 160, 71, 137, 4.0],
        [3, 84, 165, 88, 141, 4.0],
        [4, 96, 154, 101, 128, 4.0],
        [5, 106, 147, 114, 118, 4.0],
        [6, 123, 132, 132, 103, 4.0],
    ],
    columns=["id", "x1_roi", "y1_roi", "x2_roi", "y2_roi", "width_px"],
)

trail_df = manual_trails_from_roi(
    manual_trails_roi,
    roi_full_xyxy=ROI_FULL,
    downsample=DOWNSAMPLE,
    wcs=wcs,
)

if frigate_roi.size == 0 or frigate_roi.shape != (y1-y0, x1-x0):
    raise ValueError('The configured ROI does not fit inside the downsampled frame.')
if not np.isfinite(frigate_roi).all():
    raise ValueError('The measured ROI contains non-finite values.')
EXPOSURE_S = float(meta.exposure_s) if meta.exposure_s else 0.5
if not np.isfinite(EXPOSURE_S) or EXPOSURE_S <= 0:
    raise ValueError('The exposure duration is invalid.')
print('Raw shape:', frigate_full.shape, '| working shape:', frigate.shape,
      '| ROI shape:', frigate_roi.shape, '| exposure [s]:', EXPOSURE_S)


## 4. Project the validation catalogue

Apply the limiting magnitude to the MAT catalogue and project it using the solved WCS. The projected star count is calculated from these inputs.


In [ ]:
p_base = default_config('ground', 'full')
stars = load_validation_catalogue(CATALOGUE, max_mag=13.5)
projected_stars = project_catalogue_to_detector(
    stars, wcs, image_shape=frigate_full.shape,
    max_mag=13.5, max_stars=None, downsample=DOWNSAMPLE,
)
stars_in_roi = projected_stars[
    (projected_stars['x'] >= x0) & (projected_stars['x'] < x1)
    & (projected_stars['y'] >= y0) & (projected_stars['y'] < y1)
].copy()
print(len(projected_stars), 'full-frame stars;', len(stars_in_roi), 'ROI stars')


## 5. Condition the common pre-optics scene

This retains the 101-pixel median background, percentile amplitude fits, stellar multiplier 28, target multiplier 1.5, trail smoothing 1.3 pixels, and diagnostic target mask (0.1% of the aggregate peak followed by three dilations). The full-frame median filter may take several minutes. This mask is not the per-target annotation mask.


In [ ]:
# Frigate Conditioning Scene Generation
bg, resid, norm_resid, bg_sigma = background_residual(
    frigate, filter_size=101
)

star_unit = render_star_flux(
    frigate.shape, projected_stars, zero_mag_flux=1.0, psf_sigma_px=0.0
)

trail_unit = render_trail_flux(
    frigate.shape, trail_df, amplitude=1.0, sigma_px=1.3
)

star_roi_unit = star_unit[y0:y1, x0:x1]
trail_roi_unit = trail_unit[y0:y1, x0:x1]
resid_roi = resid[y0:y1, x0:x1]

star_nonzero = star_roi_unit[star_roi_unit > 0]
trail_nonzero = trail_roi_unit[trail_roi_unit > 0]
positive_resid = resid_roi[resid_roi > np.percentile(resid_roi, 90)]

STAR_SCALE = 1.0
TRAIL_SCALE = 1.0

if len(star_nonzero) > 0 and len(positive_resid) > 0:
    STAR_SCALE = np.percentile(positive_resid, 70) / max(
        np.percentile(star_nonzero, 99.5), 1e-12
    )

if len(trail_nonzero) > 0:
    TRAIL_SCALE = np.percentile(resid_roi, 99.8) / max(
        np.percentile(trail_nonzero, 99.9), 1e-12
    )

STAR_BOOST = 28.0
TRAIL_BOOST = 1.5
STAR_SCALE *= STAR_BOOST
TRAIL_SCALE *= TRAIL_BOOST

image_stars_flux = (STAR_SCALE * star_unit).astype(np.float32)
image_targets_flux = (TRAIL_SCALE * trail_unit).astype(np.float32)
image_point_flux = (image_stars_flux + image_targets_flux).astype(np.float32)
image_background_flux = bg.astype(np.float32)

target_peak = float(np.nanmax(image_targets_flux))
target_support = image_targets_flux >= max(target_peak * 1e-3, 1e-12)
target_support = binary_dilation(target_support, iterations=3)

if not np.isfinite(bg_sigma) or bg_sigma <= 0:
    raise ValueError('A positive background residual-noise estimate is required.')
validate_map(image_point_flux, 'shared source map', frigate.shape)
validate_map(image_background_flux, 'empirical background', frigate.shape)
if not target_support[y0:y1, x0:x1].any():
    raise ValueError('The diagnostic target mask is empty in the selected ROI.')
if target_support[y0:y1, x0:x1].all():
    raise ValueError('The ROI contains no pixels outside the diagnostic target mask.')

BG_MEAN_DN = float(np.nanmedian(image_background_flux))
GAIN_E_PER_DN = float(BG_MEAN_DN / max(bg_sigma**2, 1e-6))
if not np.isfinite(GAIN_E_PER_DN) or GAIN_E_PER_DN <= 0:
    raise ValueError('The effective electron-per-DN conversion gain must be positive.')
print('Effective gain [electrons/DN]:', GAIN_E_PER_DN)
print('Conditioning scales:', {'stars': float(STAR_SCALE), 'trails': float(TRAIL_SCALE)})


## 6. Run production SPOT on the shared maps

The SPOT wrapper and full working frame are retained. Exposure scaling is
handled by that wrapper. The common maps are DN-equivalent exposure signals; the native
SatSim path below does not multiply them by exposure duration a second time.


In [ ]:
# Calibrated Rendering Wrappers
fitted_psf_sigma = FITTED_PSF_SIGMA_PX


def run_spot_case(noise_on, random_seed):
    bg_mean_dn = float(np.nanmedian(image_background_flux))
    p = configure_spot_for_detector_maps(
        copy.deepcopy(p_base),
        image_shape=frigate.shape,
        exposure_s=EXPOSURE_S,
        fov_x_deg=wcs_meta.fov_x_deg,
        fov_y_deg=wcs_meta.fov_y_deg,
        deterministic=not noise_on,
        random_seed=random_seed,
        bg_mean_dn=bg_mean_dn,
        bg_sigma_dn=bg_sigma,
        psf_sigma=fitted_psf_sigma,
    )

    result = apply_spot_to_conditioned_maps(
        p,
        point_source_flux=image_point_flux,
        diffuse_background_flux=image_background_flux,
        exposure_s=EXPOSURE_S,
    )

    return np.asarray(result["Image"], dtype=np.float32), result



spot_noise_off, spot_result_off = run_spot_case(noise_on=False, random_seed=SPOT_SEED)
spot_noise_on, spot_result_on = run_spot_case(noise_on=True, random_seed=SPOT_SEED)
validate_map(spot_noise_off, 'SPOT noise-off image', frigate.shape)
validate_map(spot_noise_on, 'SPOT noise-on image', frigate.shape)
(OUT_DIR / 'spot_configuration_noise_off.json').write_text(
    json.dumps(json_safe(spot_result_off['Configuration']), indent=2, allow_nan=False),
    encoding='utf-8')
(OUT_DIR / 'spot_configuration_noise_on.json').write_text(
    json.dumps(json_safe(spot_result_on['Configuration']), indent=2, allow_nan=False),
    encoding='utf-8')
print('Production SPOT noise-off and noise-on runs completed.')


## 7. Run actual SatSim optical and photon-noise functions

The native SatSim PSF/FFT is applied once to the same source map. The shared background
is added afterwards. Expected electrons are `mean_DN × gain_e_per_DN`. Noise-off returns
the deterministic optical signal; noise-on uses **SatSim's own photon-noise function**
and divides its electron output by the same gain. No SciPy blur or NumPy Poisson draw
is used to generate either SatSim comparison image.

The integer-ADC conversion is outside this fractional-DN comparison. In particular,
SatSim's `analog_to_digital(..., dtype='float32')` still quantizes, so selecting a float
dtype is not a way to disable that step. There is no synthetic full-well clipping here.


In [ ]:
point_after_satsim_psf = satsim_convolve_map(image_point_flux)
satsim_mean_dn = point_after_satsim_psf + np.asarray(image_background_flux, np.float64)
satsim_noise_off = satsim_mean_dn.astype(np.float32)

# Match the deterministic stage boundary before introducing independent noise.
# This is a configured common-map consistency check, not a fit to the SPOT image.
stage_delta = np.asarray(spot_noise_off, np.float64) - satsim_mean_dn
stage_audit = {
    'global_max_abs_difference_dn': float(np.max(np.abs(stage_delta))),
    'roi_rmse_dn': float(np.sqrt(np.mean(stage_delta[y0:y1, x0:x1]**2))),
    'roi_max_abs_difference_dn': float(np.max(np.abs(stage_delta[y0:y1, x0:x1]))),
    'spatial_boundary': 'zero exterior support, unit-sum finite Gaussian',
    'gain_e_per_dn': GAIN_E_PER_DN,
}
(OUT_DIR / 'deterministic_stage_audit.json').write_text(
    json.dumps(stage_audit, indent=2), encoding='utf-8')
display(pd.Series(stage_audit, name='Deterministic stage audit'))

# A material difference here can indicate ADC, throughput, vignetting, PSF,
# saturation or gain/exposure settings in a different SPOT helper version.
# Do not silently compensate by shifting or rescaling either generated image.
stage_atol = 0.01
stage_rtol = 1e-5
stage_ok = np.allclose(spot_noise_off, satsim_mean_dn,
                       atol=stage_atol, rtol=stage_rtol)
if not stage_ok:
    raise RuntimeError(
        'The SPOT deterministic output does not match the configured common-map '
        'optical stage. Inspect deterministic_stage_audit.json and the saved SPOT '
        'configuration and ssa_emulator/frigate_conditioning.py before treating '
        'the two pipelines as matched. No scores have '
        'been reported and no compensating image registration/rescaling was applied.'
    )

tf.random.set_seed(SATSIM_SEED)
expected_electrons = tf.convert_to_tensor(
    satsim_mean_dn * GAIN_E_PER_DN, dtype=tf.float32)
sampled_electrons = satsim_add_photon_noise(expected_electrons, samples=None)
satsim_noise_on = (np.asarray(sampled_electrons.numpy(), np.float64)
                   / GAIN_E_PER_DN).astype(np.float32)
validate_map(satsim_noise_on, 'native SatSim noise-on image', frigate.shape)

SATSIM_NAME = 'SatSim'
SATSIM_SOURCE = 'installed SatSim native PSF, FFT convolution and photon-noise APIs'
satsim_figure8 = satsim_noise_on
spot_figure8 = spot_noise_on
FIGURE8_SATSIM_NAME = SATSIM_NAME
print('Actual SatSim functions executed successfully on the conditioned maps.')


## 8. Table IV — direct SPOT versus SatSim image consistency

The fixed ROI is compared without median alignment, resizing or fitted registration. The target-region signal is the sum of positive background-subtracted intensities in the shared diagnostic mask. Independent noise realizations are used. The table is computed afresh and need not match the earlier local approximation.


In [ ]:
# ============================================================
# 14. Generate Table IV from the Frigate-conditioned scene
# ============================================================

def table_iv_case(spot_image, satsim_image):
    x0, y0, x1, y1 = ROI_WORK

    spot_crop = np.asarray(
        spot_image[y0:y1, x0:x1],
        dtype=np.float64,
    )

    satsim_crop = np.asarray(
        satsim_image[y0:y1, x0:x1],
        dtype=np.float64,
    )

    support_crop = target_support[y0:y1, x0:x1]

    pair = direct_pair_metrics(
        satsim_crop,
        spot_crop,
    )

    satsim_flux, satsim_bg = target_integrated_flux(
        satsim_crop,
        support_crop,
    )

    spot_flux, spot_bg = target_integrated_flux(
        spot_crop,
        support_crop,
    )

    flux_difference_percent = (
        100.0
        * abs(spot_flux - satsim_flux)
        / max(abs(satsim_flux), 1e-12)
    )

    return {
        **pair,
        "SatSim_background_median_DN": satsim_bg,
        "SPOT_background_median_DN": spot_bg,
        "SatSim_target_flux_DN": satsim_flux,
        "SPOT_target_flux_DN": spot_flux,
        "Target_flux_difference_percent": flux_difference_percent,
    }

table_iv_raw = pd.DataFrame(
    {
        "Noise-off": table_iv_case(
            spot_noise_off,
            satsim_noise_off,
        ),
        "Noise-on": table_iv_case(
            spot_noise_on,
            satsim_noise_on,
        ),
    }
).T

display(table_iv_raw)

no = table_iv_raw.loc["Noise-off"]
on = table_iv_raw.loc["Noise-on"]

table_iv = pd.DataFrame(
    {
        "Metric": [
            "SSIM",
            "PSNR [dB]",
            "RMSE [DN]",
            "MAE [DN]",
            f"{SATSIM_NAME} background median [DN]",
            "SPOT background median [DN]",
            f"{SATSIM_NAME} target-region signal [DN]",
            "SPOT target-region signal [DN]",
            "Target-region signal difference [%]",
        ],
        "Noise-off": [
            f"{no['SSIM']:.4f}",
            f"{no['PSNR_dB']:.2f}",
            f"{no['RMSE_DN']:.2f}",
            f"{no['MAE_DN']:.2f}",
            f"{no['SatSim_background_median_DN']:.2f}",
            f"{no['SPOT_background_median_DN']:.2f}",
            f"{no['SatSim_target_flux_DN']:.2f}",
            f"{no['SPOT_target_flux_DN']:.2f}",
            f"{no['Target_flux_difference_percent']:.2f}",
        ],
        "Noise-on": [
            f"{on['SSIM']:.4f}",
            f"{on['PSNR_dB']:.2f}",
            f"{on['RMSE_DN']:.2f}",
            f"{on['MAE_DN']:.2f}",
            f"{on['SatSim_background_median_DN']:.2f}",
            f"{on['SPOT_background_median_DN']:.2f}",
            f"{on['SatSim_target_flux_DN']:.2f}",
            f"{on['SPOT_target_flux_DN']:.2f}",
            f"{on['Target_flux_difference_percent']:.2f}",
        ],
    }
)

display(table_iv)

table_iv_path = OUT_DIR / "Table_IV_Frigate_conditioned_SPOT_vs_satsim.csv"
table_iv.to_csv(
    table_iv_path,
    index=False,
)

table_iv_raw_path = OUT_DIR / "Table_IV_audit_raw_metrics.csv"
table_iv_raw.to_csv(
    table_iv_raw_path,
)

print("Saved:", table_iv_path)
print("Saved:", table_iv_raw_path)


## 9. Six-panel figure — Frigate, SatSim and SPOT

Each top panel has its own 1st–99.8th percentile display limits. Lower panels use the 51-pixel median background estimate and robust residual normalization, displayed from −3 to +8. These display operations do not alter the arrays used for numerical errors.


In [ ]:
# Plot Figure 8
fig, axs = plt.subplots(2, 3, figsize=(15, 8))
raw_images = [frigate, satsim_figure8, spot_figure8]
raw_titles = [
    "Frigate FITS ROI (real data)",
    f"{FIGURE8_SATSIM_NAME} matched ROI",
    "SPOT matched ROI",
]

for ax, image, title, label in zip(
    axs[0], raw_images, raw_titles, ["(a)", "(b)", "(c)"]
):
    crop = image[y0:y1, x0:x1]
    vmin, vmax = np.percentile(crop, [1, 99.8])
    ax.imshow(
        crop, cmap="gray", origin="upper", vmin=vmin, vmax=vmax
    )
    ax.set_title(title)
    ax.text(
        0.03,
        0.95,
        label,
        transform=ax.transAxes,
        va="top",
        ha="left",
        color="white",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xticks([])
    ax.set_ylabel("y pixel")
    ax.tick_params(axis="y", which="both", left=True, labelleft=True)

figure8_sigmas = {}
residual_titles = [
    "Frigate normalized residual (real data)",
    f"{FIGURE8_SATSIM_NAME} normalized residual",
    "SPOT normalized residual",
]

for ax, image, title, label, key in zip(
    axs[1],
    raw_images,
    residual_titles,
    ["(d)", "(e)", "(f)"],
    ["Frigate", FIGURE8_SATSIM_NAME, "SPOT"],
):
    crop = image[y0:y1, x0:x1]
    _, _, norm, sigma = background_residual(crop, filter_size=51)
    figure8_sigmas[key] = sigma
    ax.imshow(norm, cmap="gray", origin="upper", vmin=-3, vmax=8)
    ax.set_title(f"{title}, σ={sigma:.2f}")
    ax.text(
        0.03,
        0.95,
        label,
        transform=ax.transAxes,
        va="top",
        ha="left",
        color="white",
        fontsize=14,
        fontweight="bold",
    )
    ax.set_xlabel("x pixel")
    ax.set_ylabel("y pixel")

plt.tight_layout()
plt.savefig(
    OUT_DIR / "Figure8_Frigate_actual_SatSim_SPOT_scene_matched.png",
    dpi=400,
    bbox_inches="tight",
)
plt.savefig(OUT_DIR / 'Figure8_Frigate_actual_SatSim_SPOT_scene_matched.pdf', bbox_inches='tight')
plt.show()


## 10. Table V — comparison with measured Frigate

Whole-crop statistics remain unaligned. Only the reported median-aligned errors apply one scalar offset to each generated crop. Star counts refer to the conditioning catalogue, and the six trails are image-derived inputs; these are not independent detections in the three images.


In [ ]:
# ============================================================
# 16. Generate Table V from the SAME Figure 8 arrays
# ============================================================

frigate_crop = np.asarray(
    frigate[y0:y1, x0:x1],
    dtype=np.float64,
)

satsim_crop = np.asarray(
    satsim_figure8[y0:y1, x0:x1],
    dtype=np.float64,
)

spot_crop = np.asarray(
    spot_figure8[y0:y1, x0:x1],
    dtype=np.float64,
)

frigate_stats = {
    "median_dn": float(np.nanmedian(frigate_crop)),
    "p99_dn": float(np.nanpercentile(frigate_crop, 99)),
    "residual_sigma_dn": figure8_sigmas["Frigate"],
}

satsim_stats = frigate_reference_metrics(
    frigate_crop,
    satsim_crop,
)

spot_stats = frigate_reference_metrics(
    frigate_crop,
    spot_crop,
)

roi_size = f"{x1 - x0} x {y1 - y0}"

table_v = pd.DataFrame(
    {
        "Metric": [
            "Number of frames",
            "Selected image size [pixels]",
            "Projected catalogue stars in full frame",
            "Projected catalogue stars in selected pixels",
            "Image-derived trail signatures",
            "Whole-crop median [DN]",
            "Percentile p99 [DN]",
            "Robust residual noise [DN]",
            "Background-aligned RMSE to Frigate [DN]",
            "Background-aligned MAE to Frigate [DN]",
        ],
        "Frigate": [
            "1",
            roi_size,
            str(len(projected_stars)),
            str(len(stars_in_roi)),
            str(len(trail_df)),
            f"{frigate_stats['median_dn']:.2f}",
            f"{frigate_stats['p99_dn']:.2f}",
            f"{frigate_stats['residual_sigma_dn']:.2f}",
            "--",
            "--",
        ],
        FIGURE8_SATSIM_NAME: [
            "1",
            roi_size,
            str(len(projected_stars)),
            str(len(stars_in_roi)),
            str(len(trail_df)),
            f"{satsim_stats['median_dn']:.2f}",
            f"{satsim_stats['p99_dn']:.2f}",
            f"{satsim_stats['residual_sigma_dn']:.2f}",
            f"{satsim_stats['background_aligned_rmse_dn']:.2f}",
            f"{satsim_stats['background_aligned_mae_dn']:.2f}",
        ],
        "SPOT": [
            "1",
            roi_size,
            str(len(projected_stars)),
            str(len(stars_in_roi)),
            str(len(trail_df)),
            f"{spot_stats['median_dn']:.2f}",
            f"{spot_stats['p99_dn']:.2f}",
            f"{spot_stats['residual_sigma_dn']:.2f}",
            f"{spot_stats['background_aligned_rmse_dn']:.2f}",
            f"{spot_stats['background_aligned_mae_dn']:.2f}",
        ],
    }
)

display(table_v)

table_v_path = OUT_DIR / "Table_V_Frigate_conditioned_scene_matched.csv"

table_v.to_csv(
    table_v_path,
    index=False,
)

print("Saved:", table_v_path)

table_v_numeric = pd.DataFrame({
    'Frigate': frigate_stats, 'SatSim': satsim_stats, 'SPOT': spot_stats,
})
table_v_numeric.to_csv(OUT_DIR / 'Table_V_full_precision_metrics.csv', index_label='Metric')


## 11. Supplementary audit statistics


In [ ]:
# ============================================================
# 17. Supplementary audit statistics
# ============================================================
# These are useful for checking the run, but are not required
# in the manuscript Table V.

audit_table = pd.DataFrame(
    {
        "Metric": [
            "SSIM to Frigate",
            "PSNR to Frigate [dB]",
            "Pearson r to Frigate",
            "Raw RMSE to Frigate [DN]",
            "Raw MAE to Frigate [DN]",
            "Residual-noise ratio to Frigate",
        ],
        FIGURE8_SATSIM_NAME: [
            f"{satsim_stats['ssim_to_frigate']:.4f}",
            f"{satsim_stats['psnr_to_frigate_db']:.2f}",
            f"{satsim_stats['pearson_r_to_frigate']:.4f}",
            f"{satsim_stats['raw_rmse_dn']:.2f}",
            f"{satsim_stats['raw_mae_dn']:.2f}",
            f"{satsim_stats['residual_sigma_ratio_to_frigate']:.3f}",
        ],
        "SPOT": [
            f"{spot_stats['ssim_to_frigate']:.4f}",
            f"{spot_stats['psnr_to_frigate_db']:.2f}",
            f"{spot_stats['pearson_r_to_frigate']:.4f}",
            f"{spot_stats['raw_rmse_dn']:.2f}",
            f"{spot_stats['raw_mae_dn']:.2f}",
            f"{spot_stats['residual_sigma_ratio_to_frigate']:.3f}",
        ],
    }
)

display(audit_table)

audit_path = OUT_DIR / "Table_V_audit_statistics.csv"

audit_table.to_csv(
    audit_path,
    index=False,
)

print("Saved:", audit_path)

# Both normalized residual scales in Table V must refer to the SAME images
# and 51-pixel background procedure used in the figure.
for name, stats in [('SatSim', satsim_stats), ('SPOT', spot_stats)]:
    if not np.isclose(stats['residual_sigma_dn'], figure8_sigmas[name], rtol=1e-5, atol=1e-7):
        raise RuntimeError(
            f'{name}: the validation helper uses a different residual-noise '
            'definition from the figure. Review frigate_reference_metrics before reporting.'
        )


## 12. Save inputs, numerical arrays and provenance

Package locations, imported function source hashes and the full configuration are saved
with each run. The provenance distinguishes execution of native SatSim functions from
execution of the high-level scene generator. A package distribution version may differ
from a source checkout's version, so both are recorded when available.


In [ ]:
np.savez_compressed(
    OUT_DIR / 'validation_arrays.npz',
    Frigate=frigate, SPOT_noise_off=spot_noise_off, SPOT_noise_on=spot_noise_on,
    SatSim_noise_off=satsim_noise_off, SatSim_noise_on=satsim_noise_on,
    TargetSupport=target_support, PointInput=image_point_flux,
    StarInput=image_stars_flux, TargetInput=image_targets_flux,
    BackgroundInput=image_background_flux, SatSim_PSF=SATSIM_KERNEL,
)
projected_stars.to_csv(OUT_DIR / 'projected_stars.csv', index=False)
trail_df.to_csv(OUT_DIR / 'manual_trails.csv', index=False)

def distribution_version(name):
    try:
        return package_metadata.version(name)
    except package_metadata.PackageNotFoundError:
        return None

def source_tree_record(directory):
    directory = Path(directory)
    return {str(p.relative_to(directory)): sha256_file(p)
            for p in sorted(directory.rglob('*.py')) if '__pycache__' not in p.parts}

def git_revision(directory):
    if not (directory / '.git').exists():
        return None
    try:
        return subprocess.check_output(
            ['git', '-C', str(directory), 'rev-parse', 'HEAD'],
            stderr=subprocess.DEVNULL, text=True, timeout=5).strip()
    except (OSError, subprocess.SubprocessError):
        return None

provenance = dict(
    created_utc=datetime.now(timezone.utc).isoformat(),
    methodology='Frigate-conditioned common-map optical/photon-noise comparison',
    independent_full_forward_validation=False,
    fitted_and_evaluated_on_same_roi=True,
    satsim_executed_here=True,
    satsim_execution_scope='native optical and photon-noise APIs on shared detector maps',
    satsim_high_level_scene_generator_executed=False,
    satsim_source=SATSIM_SOURCE,
    satsim_import_path=str(SATSIM_PACKAGE_DIR),
    satsim_imported_version=getattr(satsim, '__version__', None),
    satsim_distribution_version=distribution_version('satsim'),
    satsim_git_commit=git_revision(SATSIM_PACKAGE_DIR.parent),
    satsim_functions=native_api_provenance,
    satsim_source_sha256=source_tree_record(SATSIM_PACKAGE_DIR),
    environment={
        'python': sys.version, 'executable': sys.executable, 'platform': platform.platform(),
        'tensorflow': tf.__version__, 'numpy': np.__version__,
        'pandas': pd.__version__, 'scipy': distribution_version('scipy'),
        'astropy': distribution_version('astropy'), 'use_cpu': USE_CPU,
    },
    roi_full=ROI_FULL, roi_work=ROI_WORK, downsample=DOWNSAMPLE,
    downsample_method='arithmetic block mean; incomplete edge blocks discarded',
    exposure_s=EXPOSURE_S,
    star_scale=STAR_SCALE, trail_scale=TRAIL_SCALE,
    star_boost=STAR_BOOST, trail_boost=TRAIL_BOOST,
    background_sigma=bg_sigma, gain_e_per_dn=GAIN_E_PER_DN,
    fitted_psf_sigma=FITTED_PSF_SIGMA_PX,
    satsim_psf=satsim_psf_settings,
    satsim_noise={'photon_shot_noise': True, 'poisson_samples': 1,
                  'read_noise_e': 0.0, 'dark_current_e': 0.0, 'bias_dn': 0.0},
    satsim_output_domain='fractional DN; outside integer ADC and saturation',
    seeds={'SPOT': SPOT_SEED, 'SatSim_TensorFlow': SATSIM_SEED},
    native_smoke_results=native_smoke_results,
    deterministic_stage_audit=stage_audit,
    deterministic_stage_check={'passed': bool(stage_ok), 'atol_dn': stage_atol,
                               'rtol': stage_rtol},
    target_signal_diagnostic={'relative_aggregate_threshold': 0.001, 'dilations': 3,
                              'statistic': 'sum of positive background-subtracted pixels'},
    spot_config_noise_off=spot_result_off['Configuration'],
    spot_config_noise_on=spot_result_on['Configuration'],
    input_sha256={p.name: sha256_file(p) for p in (RAW_FITS, wcs_path, CATALOGUE)},
    spot_source_sha256=source_tree_record(ROOT / 'ssa_emulator'),
)
(OUT_DIR / 'provenance.json').write_text(
    json.dumps(json_safe(provenance), indent=2, allow_nan=False), encoding='utf-8')
(OUT_DIR / 'native_satsim_checks.json').write_text(
    json.dumps(native_smoke_results, indent=2), encoding='utf-8')

# Save a version snapshot in addition to the hashes (no dependency changes).
(OUT_DIR / 'environment_versions.json').write_text(
    json.dumps({name: distribution_version(name) for name in
                ('satsim', 'tensorflow', 'tensorflow-cpu', 'numpy', 'scipy',
                 'pandas', 'astropy', 'matplotlib', 'scikit-image')}, indent=2),
    encoding='utf-8')
print('Saved new arrays, tables, figure and provenance to:', OUT_DIR)


## Interpretation and reproducibility

The tables describe a comparison of native SatSim optical and photon-noise routines with SPOT under common Frigate-conditioned source maps, background, PSF and effective-gain settings. Images are evaluated in fractional DN before integer quantization. This notebook does not run catalogue/orbit-driven SatSim scene generation.

Retain package versions, source hashes, run configurations and numerical arrays alongside any reported results. If the deterministic consistency check fails, inspect `deterministic_stage_audit.json`, the saved SPOT configuration and `ssa_emulator/frigate_conditioning.py` before interpreting the comparison.

Reference APIs: [SatSim repository](https://github.com/ssc-ai/satsim), [image modules](https://satsim.readthedocs.io/en/latest/api/satsim.image.html), [math modules](https://satsim.readthedocs.io/en/latest/api/satsim.math.html).
